In [ ]:
# --- repository layout: src/ and run/ go on the import path, and relative paths
# --- such as "data/..." are resolved from the repository root.
import os as _os
import sys as _sys
from pathlib import Path as _Path

_HERE = _Path(__file__).resolve().parent if "__file__" in globals() else _Path.cwd()
_ROOT = next(p for p in (_HERE, *_HERE.parents) if (p / "src").is_dir() and (p / "run").is_dir())
for _p in (_ROOT / "run", _ROOT / "src", _ROOT / "src" / "quantum_hardware", _HERE):
    if str(_p) not in _sys.path:
        _sys.path.insert(0, str(_p))
_os.chdir(_ROOT)
# ---------------------------------------------------------------------------------


In [ ]:



# ======================================================================
# Exact CNOT-cosine HPINN: random-collocation sweep
# ======================================================================
# This experiment is intentionally separate from the original calculation
# above. It compares five fixed CNOT topologies and depths 1, 2, and 3 while
# reproducing the colleague's training protocol and collocation sweep.
import csv
import gc
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from main.functions import (
    predict,
    reference_at_time,
    reference_pde_residual,
    relative_l2_error,
    sample_collocation_points,
    set_seed,
    solve_burgers_fd,
    train,
)
from main.models import QuantumExactPINN

# Domain and reference-solver constants, matching the defaults shared by
# run_classical.py, run_quantum.py, and run_qubit_sweep.py.
L, NU = 1.0, 0.01
HX, HT = L / 512, 1.0e-3


# Architecture-matched quantum configuration. N_QUBITS=10 replaces the
# ten-dimensional classical internal interface. Pre and post blocks retain
# ten neurons and one hidden layer each.
QEXACT_N_QUBITS = 10
QEXACT_PRE_NEURONS = 10
QEXACT_PRE_DEPTH = 1
QEXACT_POST_NEURONS = 10
QEXACT_POST_DEPTH = 1
QEXACT_INPUT_ROTATION_AXIS = "rx"
QEXACT_ANGLE_SCALE = None

QEXACT_CNOT_NETWORKS = (
    "linear_cnot",
    "reverse_linear_cnot",
    "ring_cnot",
    "brickwork_cnot",
    "symmetric_cnot_ladder",
)
QEXACT_CNOT_LAYER_VALUES = (1, 2, 3)
QEXACT_TFIN_VALUES = (0.1, 1.0)
QEXACT_N_COLLOCATION_VALUES = (
    10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 250, 500,
)
QEXACT_SEEDS = tuple(range(5))

# Training settings reproduced from the supplied protocol.
QEXACT_ADAM_EPOCHS = 20_000
QEXACT_ADAM_LR = 1.0e-3
QEXACT_WEIGHT_DECAY = 1.0e-6
QEXACT_GRAD_CLIP = 1.0
QEXACT_LBFGS_EPOCHS = 1_000
QEXACT_LBFGS_LR = 0.5
QEXACT_LBFGS_PATIENCE = 50
QEXACT_LBFGS_TOL = 1.0e-9
QEXACT_COLLOCATION_DISTRIBUTION = "random"
QEXACT_N_EVALUATION_POINTS = 200

# Set False to regenerate summaries and plots from existing raw JSON files
# without training missing cases. Completed cases are always skipped safely.
QEXACT_TRAIN_MISSING_CASES = True

# Optional smoke-test limiter. None runs all missing cases; an integer such as
# 1 or 5 stops after that many new trainings while preserving their results.
QEXACT_MAX_NEW_RUNS = None

# Results live beside, not inside, the CBPF geral source directory. Set this
# override to a path string only when the Jupyter kernel was started elsewhere
# and cannot infer the project directory from its current working directory.
QEXACT_PROJECT_DIRECTORY_OVERRIDE = None


def _qexact_resolve_project_directory():
    """Resolve the source root both in a .py script and in a Jupyter cell."""
    if QEXACT_PROJECT_DIRECTORY_OVERRIDE is not None:
        project_directory = Path(QEXACT_PROJECT_DIRECTORY_OVERRIDE).expanduser().resolve()
        if not project_directory.is_dir():
            raise FileNotFoundError(
                "QEXACT_PROJECT_DIRECTORY_OVERRIDE does not identify an existing directory: "
                f"{project_directory}"
            )
        return project_directory

    # Normal execution through `python run_quantumExact.py` or `%run`.
    if "__file__" in globals():
        return Path(globals()["__file__"]).resolve().parent

    # Direct execution of the code in a Jupyter cell.
    working_directory = Path.cwd().resolve()
    if working_directory.name.casefold() == "cbpf geral":
        return working_directory

    child_project = working_directory / "CBPF geral"
    if child_project.is_dir():
        return child_project.resolve()

    # This last fallback supports a notebook stored directly in the project
    # under a different folder name; the resolved paths are printed at startup.
    return working_directory


QEXACT_PROJECT_DIRECTORY = _qexact_resolve_project_directory()
QEXACT_RESULTS_DIRECTORY = (
    QEXACT_PROJECT_DIRECTORY.parent / "CBPF_quantum_exact_results"
)
QEXACT_RAW_DIRECTORY = QEXACT_RESULTS_DIRECTORY / "raw_runs"
QEXACT_SUMMARY_DIRECTORY = QEXACT_RESULTS_DIRECTORY / "summaries"
QEXACT_PLOT_DIRECTORY = QEXACT_RESULTS_DIRECTORY / "plots"
QEXACT_RUN_TABLE_PATH = QEXACT_RESULTS_DIRECTORY / "quantum_exact_runs.csv"
QEXACT_SUMMARY_TABLE_PATH = QEXACT_RESULTS_DIRECTORY / "quantum_exact_summary.csv"


def _qexact_float_token(value):
    """Filesystem-safe, stable representation of values such as 0.1 and 1.0."""
    return f"{float(value):g}".replace("-", "m").replace(".", "p")


def _qexact_run_path(network, n_layers, Tfin, n_collocation, seed):
    return (
        QEXACT_RAW_DIRECTORY
        / network
        / f"layers_{int(n_layers)}"
        / f"T_{_qexact_float_token(Tfin)}"
        / f"N_{int(n_collocation)}"
        / f"seed_{int(seed)}.json"
    )


def _qexact_atomic_json_dump(payload, path):
    """Write one completed run atomically so interrupted runs are retrained."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, sort_keys=True)
    temporary_path.replace(path)


def _qexact_load_run_rows():
    """Load every completed run, ignoring incomplete temporary files."""
    rows = []
    if not QEXACT_RAW_DIRECTORY.exists():
        return rows
    for path in sorted(QEXACT_RAW_DIRECTORY.rglob("seed_*.json")):
        try:
            with path.open("r", encoding="utf-8") as handle:
                row = json.load(handle)
        except (OSError, json.JSONDecodeError) as error:
            print(f"Warning: ignoring unreadable result {path}: {error}")
            continue
        rows.append(row)
    return rows


QEXACT_RUN_FIELDS = (
    "network",
    "n_cnot_layers",
    "Tfin",
    "n_collocation",
    "seed",
    "mse",
    "mae",
    "relative_l2",
    "target_loss",
    "target_reached",
    "stopped_at_step",
    "adam_steps",
    "lbfgs_steps",
    "elapsed_seconds",
    "trainable_parameters",
)


def _qexact_atomic_csv_dump(rows, path, fieldnames):
    """Rewrite a compact aggregate table atomically from the raw run files."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")
    with temporary_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
    temporary_path.replace(path)


def _qexact_summarize(rows):
    """Aggregate MSE statistics over seeds for every plotted point."""
    grouped = {}
    for row in rows:
        key = (
            str(row["network"]),
            int(row["n_cnot_layers"]),
            float(row["Tfin"]),
            int(row["n_collocation"]),
        )
        grouped.setdefault(key, []).append(float(row["mse"]))

    summary = []
    for (network, n_layers, Tfin, n_collocation), values in sorted(grouped.items()):
        values = np.asarray(values, dtype=np.float64)
        summary.append(
            {
                "network": network,
                "n_cnot_layers": n_layers,
                "Tfin": Tfin,
                "n_collocation": n_collocation,
                "mse_mean": float(np.mean(values)),
                "mse_min": float(np.min(values)),
                "mse_max": float(np.max(values)),
                "mse_std": float(np.std(values)),
                "n_samples": int(values.size),
            }
        )
    return summary


QEXACT_SUMMARY_FIELDS = (
    "network",
    "n_cnot_layers",
    "Tfin",
    "n_collocation",
    "mse_mean",
    "mse_min",
    "mse_max",
    "mse_std",
    "n_samples",
)


def _qexact_save_summary_data_and_plots(rows):
    """Generate the 30 requested figures and their compact source arrays."""
    rows = sorted(
        rows,
        key=lambda row: (
            row["network"],
            int(row["n_cnot_layers"]),
            float(row["Tfin"]),
            int(row["n_collocation"]),
            int(row["seed"]),
        ),
    )
    _qexact_atomic_csv_dump(rows, QEXACT_RUN_TABLE_PATH, QEXACT_RUN_FIELDS)

    summary = _qexact_summarize(rows)
    _qexact_atomic_csv_dump(
        summary,
        QEXACT_SUMMARY_TABLE_PATH,
        QEXACT_SUMMARY_FIELDS,
    )
    QEXACT_SUMMARY_DIRECTORY.mkdir(parents=True, exist_ok=True)
    QEXACT_PLOT_DIRECTORY.mkdir(parents=True, exist_ok=True)

    for network in QEXACT_CNOT_NETWORKS:
        for n_layers in QEXACT_CNOT_LAYER_VALUES:
            for Tfin in QEXACT_TFIN_VALUES:
                selected = [
                    row
                    for row in summary
                    if row["network"] == network
                    and int(row["n_cnot_layers"]) == int(n_layers)
                    and np.isclose(float(row["Tfin"]), float(Tfin))
                ]
                if not selected:
                    continue
                selected.sort(key=lambda row: int(row["n_collocation"]))

                n_collocation = np.asarray(
                    [row["n_collocation"] for row in selected], dtype=np.int64
                )
                mse_mean = np.asarray(
                    [row["mse_mean"] for row in selected], dtype=np.float64
                )
                mse_min = np.asarray(
                    [row["mse_min"] for row in selected], dtype=np.float64
                )
                mse_max = np.asarray(
                    [row["mse_max"] for row in selected], dtype=np.float64
                )
                mse_std = np.asarray(
                    [row["mse_std"] for row in selected], dtype=np.float64
                )
                n_samples = np.asarray(
                    [row["n_samples"] for row in selected], dtype=np.int64
                )

                file_stem = (
                    f"mse_{network}_layers_{n_layers}"
                    f"_T_{_qexact_float_token(Tfin)}"
                )
                np.savez_compressed(
                    QEXACT_SUMMARY_DIRECTORY / f"{file_stem}.npz",
                    n_collocation=n_collocation,
                    mse_mean=mse_mean,
                    mse_min=mse_min,
                    mse_max=mse_max,
                    mse_std=mse_std,
                    n_samples=n_samples,
                    network=network,
                    n_cnot_layers=int(n_layers),
                    Tfin=float(Tfin),
                )

                positions = np.arange(n_collocation.size)
                asymmetric_errors = np.vstack(
                    (
                        np.maximum(0.0, mse_mean - mse_min),
                        np.maximum(0.0, mse_max - mse_mean),
                    )
                )

                fig, ax = plt.subplots(figsize=(9.0, 4.8))
                ax.errorbar(
                    positions,
                    mse_mean,
                    yerr=asymmetric_errors,
                    fmt="o-",
                    color="#2878c8",
                    linewidth=2.0,
                    markersize=5.5,
                    capsize=4,
                    label="MSE mean (min-max over seeds)",
                )
                ax.set_xlabel("N_COLLOCATION", fontsize=13)
                ax.set_ylabel("MSE", fontsize=13)
                ax.set_yscale("log")
                ax.set_xticks(positions)
                ax.set_xticklabels([str(value) for value in n_collocation])
                ax.set_title(
                    f"Exact CNOT-cosine HPINN: {network}, "
                    f"{n_layers} layer(s), T = {Tfin:g}",
                    fontsize=12,
                )
                ax.grid(True, which="both", linestyle="--", alpha=0.45)
                ax.legend(fontsize=9)
                fig.tight_layout()
                fig.savefig(
                    QEXACT_PLOT_DIRECTORY / f"{file_stem}.png",
                    dpi=250,
                    bbox_inches="tight",
                )
                plt.close(fig)


def run_quantum_exact_collocation_sweep(
    train_missing=QEXACT_TRAIN_MISSING_CASES,
    max_new_runs=QEXACT_MAX_NEW_RUNS,
):
    """Run or resume all exact-quantum cases, then regenerate tables/plots."""
    for directory in (
        QEXACT_RAW_DIRECTORY,
        QEXACT_SUMMARY_DIRECTORY,
        QEXACT_PLOT_DIRECTORY,
    ):
        directory.mkdir(parents=True, exist_ok=True)

    total_cases = (
        len(QEXACT_CNOT_NETWORKS)
        * len(QEXACT_CNOT_LAYER_VALUES)
        * len(QEXACT_TFIN_VALUES)
        * len(QEXACT_N_COLLOCATION_VALUES)
        * len(QEXACT_SEEDS)
    )
    completed_paths = {
        path.resolve() for path in QEXACT_RAW_DIRECTORY.rglob("seed_*.json")
    }
    print(f"Exact-quantum sweep: {len(completed_paths)}/{total_cases} runs already saved.")
    print(f"Results directory: {QEXACT_RESULTS_DIRECTORY}")

    if not train_missing:
        rows = _qexact_load_run_rows()
        _qexact_save_summary_data_and_plots(rows)
        return rows

    # The expensive FDM reference is computed only once for each final time.
    references = {}
    for Tfin in QEXACT_TFIN_VALUES:
        x_ref, t_ref, U_ref = solve_burgers_fd(L=L, Tfin=Tfin, nu=NU)
        x_plot = np.linspace(
            0.0,
            L,
            QEXACT_N_EVALUATION_POINTS,
        ).reshape(-1, 1)
        u_ref_eval, _ = reference_at_time(
            Tfin,
            t_ref,
            x_ref,
            U_ref,
            x_plot,
            L=L,
        )
        references[float(Tfin)] = (
            x_ref,
            t_ref,
            U_ref,
            x_plot,
            u_ref_eval,
        )

    new_runs = 0
    for network in QEXACT_CNOT_NETWORKS:
        for n_layers in QEXACT_CNOT_LAYER_VALUES:
            for Tfin in QEXACT_TFIN_VALUES:
                x_ref, t_ref, U_ref, x_plot, u_ref_eval = references[float(Tfin)]

                for n_collocation in QEXACT_N_COLLOCATION_VALUES:
                    for seed in QEXACT_SEEDS:
                        result_path = _qexact_run_path(
                            network,
                            n_layers,
                            Tfin,
                            n_collocation,
                            seed,
                        )
                        if result_path.exists():
                            continue

                        if max_new_runs is not None and new_runs >= max_new_runs:
                            rows = _qexact_load_run_rows()
                            _qexact_save_summary_data_and_plots(rows)
                            print(f"Smoke-test limit reached after {new_runs} new run(s).")
                            return rows

                        print(
                            f"\n[{len(completed_paths) + new_runs + 1}/{total_cases}] "
                            f"network={network} | layers={n_layers} | "
                            f"T={Tfin:g} | Nf={n_collocation} | seed={seed}"
                        )

                        # Resetting immediately before construction pairs both
                        # initialization and collocation streams across circuits.
                        set_seed(seed)
                        model = QuantumExactPINN(
                            n_qubits=QEXACT_N_QUBITS,
                            pre_neurons=QEXACT_PRE_NEURONS,
                            pre_depth=QEXACT_PRE_DEPTH,
                            post_neurons=QEXACT_POST_NEURONS,
                            post_depth=QEXACT_POST_DEPTH,
                            fixed_layer=network,
                            fixed_layer_repeats=n_layers,
                            input_rotation_axis=QEXACT_INPUT_ROTATION_AXIS,
                            angle_scale=QEXACT_ANGLE_SCALE,
                            L=L,
                            Tfin=Tfin,
                        )

                        x_f, t_f = sample_collocation_points(
                            n_collocation,
                            L,
                            Tfin,
                            HT,
                            seed=seed,
                            distribution=QEXACT_COLLOCATION_DISTRIBUTION,
                        )

                        # AdamW receives a newly sampled random set every epoch.
                        # L-BFGS keeps the last AdamW set fixed during each line
                        # search, as required for a deterministic local objective.
                        resample_fn = lambda n=n_collocation, tf=Tfin: (
                            sample_collocation_points(
                                n,
                                L,
                                tf,
                                HT,
                                seed=None,
                                distribution="random",
                            )
                        )

                        target_loss = reference_pde_residual(
                            x_f,
                            t_f,
                            x_ref,
                            t_ref,
                            U_ref,
                            L=L,
                            nu=NU,
                            hx=HX,
                            ht=HT,
                        )

                        start_time = time.time()
                        model, hist = train(
                            model,
                            x_f,
                            t_f,
                            loss_kwargs=dict(
                                L=L,
                                Tfin=Tfin,
                                nu=NU,
                                hx=HX,
                                ht=HT,
                            ),
                            epochs=QEXACT_ADAM_EPOCHS,
                            lr=QEXACT_ADAM_LR,
                            grad_clip=QEXACT_GRAD_CLIP,
                            weight_decay=QEXACT_WEIGHT_DECAY,
                            resample_fn=resample_fn,
                            lbfgs_epochs=QEXACT_LBFGS_EPOCHS,
                            lbfgs_lr=QEXACT_LBFGS_LR,
                            lbfgs_resample_fn=None,
                            lbfgs_patience=QEXACT_LBFGS_PATIENCE,
                            lbfgs_tol=QEXACT_LBFGS_TOL,
                            target_loss=target_loss,
                        )
                        elapsed_seconds = time.time() - start_time

                        u_pred = predict(model, x_plot, Tfin)
                        mse = float(np.mean((u_pred - u_ref_eval) ** 2))
                        mae = float(np.mean(np.abs(u_pred - u_ref_eval)))
                        rel_l2 = float(relative_l2_error(u_pred, u_ref_eval))
                        stopped_at_step = int(len(hist["loss"]))
                        adam_steps = min(stopped_at_step, QEXACT_ADAM_EPOCHS)
                        lbfgs_steps = max(0, stopped_at_step - QEXACT_ADAM_EPOCHS)

                        payload = {
                            "network": network,
                            "n_cnot_layers": int(n_layers),
                            "Tfin": float(Tfin),
                            "n_collocation": int(n_collocation),
                            "seed": int(seed),
                            "mse": mse,
                            "mae": mae,
                            "relative_l2": rel_l2,
                            "target_loss": float(target_loss),
                            "target_reached": bool(hist["target_reached"]),
                            "stopped_at_step": stopped_at_step,
                            "adam_steps": int(adam_steps),
                            "lbfgs_steps": int(lbfgs_steps),
                            "elapsed_seconds": float(elapsed_seconds),
                            "trainable_parameters": int(
                                sum(
                                    parameter.numel()
                                    for parameter in model.parameters()
                                    if parameter.requires_grad
                                )
                            ),
                            "model_config": dict(model.config),
                            "training_config": {
                                "adam_epochs": QEXACT_ADAM_EPOCHS,
                                "adam_lr": QEXACT_ADAM_LR,
                                "weight_decay": QEXACT_WEIGHT_DECAY,
                                "grad_clip": QEXACT_GRAD_CLIP,
                                "lbfgs_epochs": QEXACT_LBFGS_EPOCHS,
                                "lbfgs_lr": QEXACT_LBFGS_LR,
                                "lbfgs_patience": QEXACT_LBFGS_PATIENCE,
                                "lbfgs_tol": QEXACT_LBFGS_TOL,
                                "collocation_distribution": QEXACT_COLLOCATION_DISTRIBUTION,
                                "resample_each_adam_epoch": True,
                                "n_evaluation_points": QEXACT_N_EVALUATION_POINTS,
                                "L": float(L),
                                "nu": float(NU),
                                "hx": float(HX),
                                "ht": float(HT),
                            },
                        }
                        _qexact_atomic_json_dump(payload, result_path)
                        new_runs += 1

                        print(
                            f"saved | MSE={mse:.6e} | target={target_loss:.3e} | "
                            f"steps={stopped_at_step} | time={elapsed_seconds:.1f}s"
                        )

                        del model, hist, x_f, t_f, u_pred
                        gc.collect()

    rows = _qexact_load_run_rows()
    _qexact_save_summary_data_and_plots(rows)
    print(f"Completed raw runs: {len(rows)}/{total_cases}")
    print(f"Plots saved in: {QEXACT_PLOT_DIRECTORY}")
    return rows


# NOTE: the saved notebook guarded this call with an undefined
# `RUN_QUANTUM_EXACT_COLLOCATION_SWEEP` flag (a NameError waiting to happen,
# since __name__ == "__main__" is always true inside a Jupyter kernel). The
# sibling scripts (run_classical.py, run_quantum.py, ...) all just use a bare
# `if __name__ == "__main__":` guard, so this does the same. Set
# QEXACT_TRAIN_MISSING_CASES = False above to only rebuild summaries/plots
# from existing raw JSON files without training anything new.
if __name__ == "__main__":
    run_quantum_exact_collocation_sweep()


In [8]:
"""Regenerate comparison-only plots from saved exact-HPINN runs.

This script does not train a model and does not import or modify
``run_quantumExact.py``.  It reads the individual results already stored in
``CBPF_quantum_exact_results/raw_runs`` (or, as a fallback, the compact
``quantum_exact_runs.csv`` table), aggregates the five seeds, and creates:

1. MSE versus N_COLLOCATION, with the population variance passed directly as
   ``yerr``.  This intentionally reproduces the colleague's plotting
   convention for a like-for-like visual comparison.  Variance is not a
   statistically standard uncertainty-bar length because it has squared
   units.
2. Mean stopping step and population variance of stopping steps on two y axes,
   following the supplied example.

All generated files are written to a separate sibling directory named
``CBPF_quantum_exact_colleague_comparison``.  Existing scientific plots in
``CBPF_quantum_exact_results`` are left untouched.
"""

from __future__ import annotations

import csv
import json
from collections import defaultdict
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np


# ---------------------------------------------------------------------------
# User configuration
# ---------------------------------------------------------------------------

# Use a path string when the automatic project-directory detection is not
# appropriate in Jupyter.  The path must identify the folder "CBPF geral".
PROJECT_DIRECTORY_OVERRIDE = None

# Optional path overrides.  Leave as None to use sibling folders of
# PROJECT_DIRECTORY.
SOURCE_RESULTS_DIRECTORY_OVERRIDE = None
OUTPUT_DIRECTORY_OVERRIDE = None

# Both final times are placed side by side, as in the colleague's figures.
TFIN_VALUES = (0.1, 1.0)

# Leave as None to process every network/layer value found in the saved data.
# Examples: NETWORK_FILTER = ("ring_cnot",)
#           CNOT_LAYER_FILTER = (1, 2, 3)
NETWORK_FILTER = None
CNOT_LAYER_FILTER = None

# Population variance (ddof=0) reproduces numpy.var(...) in the colleague's
# implementation.  Change to 1 only if sample variance is explicitly desired.
VARIANCE_DDOF = 0

SAVE_PDF = True
PLOT_DPI = 250


# ---------------------------------------------------------------------------
# Path handling and data loading
# ---------------------------------------------------------------------------

def resolve_project_directory() -> Path:
    """Resolve the source folder in normal Python and direct Jupyter use."""
    if PROJECT_DIRECTORY_OVERRIDE is not None:
        project = Path(PROJECT_DIRECTORY_OVERRIDE).expanduser().resolve()
        if not project.is_dir():
            raise FileNotFoundError(
                "PROJECT_DIRECTORY_OVERRIDE is not an existing directory: "
                f"{project}"
            )
        return project

    if "__file__" in globals():
        return Path(globals()["__file__"]).resolve().parent

    working_directory = Path.cwd().resolve()
    if working_directory.name.casefold() == "cbpf geral":
        return working_directory

    candidate = working_directory / "CBPF geral"
    if candidate.is_dir():
        return candidate.resolve()

    return working_directory


PROJECT_DIRECTORY = resolve_project_directory()

SOURCE_RESULTS_DIRECTORY = (
    Path(SOURCE_RESULTS_DIRECTORY_OVERRIDE).expanduser().resolve()
    if SOURCE_RESULTS_DIRECTORY_OVERRIDE is not None
    else PROJECT_DIRECTORY.parent / "CBPF_quantum_exact_results_uniform"
)

OUTPUT_DIRECTORY = (
    Path(OUTPUT_DIRECTORY_OVERRIDE).expanduser().resolve()
    if OUTPUT_DIRECTORY_OVERRIDE is not None
    else PROJECT_DIRECTORY.parent / "CBPF_quantum_exact_colleague_comparison_uniform"
)

RAW_DIRECTORY = SOURCE_RESULTS_DIRECTORY / "raw_runs"
RUN_TABLE_PATH = SOURCE_RESULTS_DIRECTORY / "quantum_exact_runs.csv"
SUMMARY_DIRECTORY = OUTPUT_DIRECTORY / "summaries"
MSE_PLOT_DIRECTORY = OUTPUT_DIRECTORY / "mse_plots"
STEP_PLOT_DIRECTORY = OUTPUT_DIRECTORY / "step_plots"
SUMMARY_TABLE_PATH = SUMMARY_DIRECTORY / "colleague_style_summary.csv"


REQUIRED_FIELDS = {
    "network",
    "n_cnot_layers",
    "Tfin",
    "n_collocation",
    "seed",
    "mse",
    "stopped_at_step",
}


def load_saved_rows() -> list[dict]:
    """Load raw JSON results, falling back to the consolidated run table."""
    rows: list[dict] = []

    if RAW_DIRECTORY.is_dir():
        for path in sorted(RAW_DIRECTORY.rglob("seed_*.json")):
            try:
                with path.open("r", encoding="utf-8") as handle:
                    rows.append(json.load(handle))
            except (OSError, json.JSONDecodeError) as error:
                print(f"Warning: ignoring unreadable result {path}: {error}")

    if not rows and RUN_TABLE_PATH.is_file():
        with RUN_TABLE_PATH.open("r", encoding="utf-8", newline="") as handle:
            rows.extend(csv.DictReader(handle))

    if not rows:
        raise FileNotFoundError(
            "No saved run data were found. Expected raw JSON files below "
            f"'{RAW_DIRECTORY}' or the table '{RUN_TABLE_PATH}'."
        )

    missing = REQUIRED_FIELDS.difference(rows[0])
    if missing:
        raise ValueError(
            "The saved records do not contain the required columns: "
            + ", ".join(sorted(missing))
        )

    return rows


# ---------------------------------------------------------------------------
# Aggregation
# ---------------------------------------------------------------------------

def aggregate_rows(rows: list[dict]) -> list[dict]:
    """Aggregate MSE and stopping steps over seeds for each configuration."""
    grouped: dict[tuple, dict[str, list[float]]] = defaultdict(
        lambda: {"mse": [], "steps": [], "seeds": []}
    )

    for row in rows:
        network = str(row["network"])
        n_layers = int(row["n_cnot_layers"])

        if NETWORK_FILTER is not None and network not in NETWORK_FILTER:
            continue
        if CNOT_LAYER_FILTER is not None and n_layers not in CNOT_LAYER_FILTER:
            continue

        key = (
            network,
            n_layers,
            float(row["Tfin"]),
            int(row["n_collocation"]),
        )
        grouped[key]["mse"].append(float(row["mse"]))
        grouped[key]["steps"].append(float(row["stopped_at_step"]))
        grouped[key]["seeds"].append(int(row["seed"]))

    summary: list[dict] = []
    for key, values in sorted(grouped.items()):
        network, n_layers, Tfin, n_collocation = key
        mse = np.asarray(values["mse"], dtype=np.float64)
        steps = np.asarray(values["steps"], dtype=np.float64)

        if mse.size <= VARIANCE_DDOF:
            mse_variance = np.nan
            steps_variance = np.nan
        else:
            mse_variance = float(np.var(mse, ddof=VARIANCE_DDOF))
            steps_variance = float(np.var(steps, ddof=VARIANCE_DDOF))

        summary.append(
            {
                "network": network,
                "n_cnot_layers": n_layers,
                "Tfin": Tfin,
                "n_collocation": n_collocation,
                "mse_mean": float(np.mean(mse)),
                "mse_variance": mse_variance,
                "steps_mean": float(np.mean(steps)),
                "steps_variance": steps_variance,
                "n_samples": int(mse.size),
                "seeds": ",".join(str(seed) for seed in sorted(values["seeds"])),
            }
        )

    if not summary:
        raise ValueError("No records remain after applying the configured filters.")

    return summary


def save_summary(summary: list[dict]) -> None:
    """Save the exact values used to construct the comparison plots."""
    SUMMARY_DIRECTORY.mkdir(parents=True, exist_ok=True)
    fieldnames = list(summary[0].keys())
    with SUMMARY_TABLE_PATH.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary)


# ---------------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------------

def float_token(value: float) -> str:
    return f"{float(value):g}".replace("-", "m").replace(".", "p")


def rows_for_configuration(
    summary: list[dict], network: str, n_layers: int, Tfin: float
) -> list[dict]:
    selected = [
        row
        for row in summary
        if row["network"] == network
        and int(row["n_cnot_layers"]) == n_layers
        and np.isclose(float(row["Tfin"]), Tfin)
    ]
    return sorted(selected, key=lambda row: int(row["n_collocation"]))


def save_figure(fig: plt.Figure, directory: Path, stem: str) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        directory / f"{stem}.png",
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    if SAVE_PDF:
        fig.savefig(directory / f"{stem}.pdf", bbox_inches="tight")
    plt.close(fig)


def plot_mse_comparison(
    summary: list[dict], network: str, n_layers: int
) -> bool:
    """Plot mean MSE with variance used directly as yerr."""
    figure_rows = [
        rows_for_configuration(summary, network, n_layers, Tfin)
        for Tfin in TFIN_VALUES
    ]
    if not any(figure_rows):
        return False

    fig, axes = plt.subplots(
        1,
        len(TFIN_VALUES),
        figsize=(7.2 * len(TFIN_VALUES), 5.0),
        squeeze=False,
    )
    axes = axes.ravel()

    for ax, Tfin, selected in zip(axes, TFIN_VALUES, figure_rows):
        if not selected:
            ax.set_axis_off()
            ax.set_title(f"T = {Tfin:g}: no saved data")
            continue

        n_collocation = np.asarray(
            [row["n_collocation"] for row in selected], dtype=np.int64
        )
        mse_mean = np.asarray(
            [row["mse_mean"] for row in selected], dtype=np.float64
        )
        mse_variance = np.asarray(
            [row["mse_variance"] for row in selected], dtype=np.float64
        )
        positions = np.arange(n_collocation.size)

        ax.errorbar(
            positions,
            mse_mean,
            yerr=mse_variance,
            fmt="o-",
            color="#2878c8",
            linewidth=1.8,
            markersize=5.0,
            capsize=4,
            label="MSE (mean +/- variance)",
        )
        ax.set_yscale("log")
        ax.set_xticks(positions)
        ax.set_xticklabels([str(value) for value in n_collocation])
        ax.set_xlabel("N_COLLOCATION")
        ax.set_ylabel("MSE")
        ax.set_title(f"T = {Tfin:g}")
        ax.grid(True, which="both", linestyle="--", alpha=0.45)
        ax.legend(fontsize=9)

    fig.suptitle(
        f"Exact CNOT-cosine HPINN: {network}, {n_layers} layer(s)",
        fontsize=14,
    )
    fig.tight_layout()
    save_figure(
        fig,
        MSE_PLOT_DIRECTORY,
        f"MSE_vs_N_COLLOCATION_{network}_layers_{n_layers}",
    )
    return True


def plot_step_comparison(
    summary: list[dict], network: str, n_layers: int
) -> bool:
    """Plot mean stopping step and step variance on separate y axes."""
    figure_rows = [
        rows_for_configuration(summary, network, n_layers, Tfin)
        for Tfin in TFIN_VALUES
    ]
    if not any(figure_rows):
        return False

    fig, axes = plt.subplots(
        1,
        len(TFIN_VALUES),
        figsize=(7.2 * len(TFIN_VALUES), 5.0),
        squeeze=False,
    )
    axes = axes.ravel()

    for ax, Tfin, selected in zip(axes, TFIN_VALUES, figure_rows):
        if not selected:
            ax.set_axis_off()
            ax.set_title(f"T = {Tfin:g}: no saved data")
            continue

        n_collocation = np.asarray(
            [row["n_collocation"] for row in selected], dtype=np.int64
        )
        steps_mean = np.asarray(
            [row["steps_mean"] for row in selected], dtype=np.float64
        )
        steps_variance = np.asarray(
            [row["steps_variance"] for row in selected], dtype=np.float64
        )
        positions = np.arange(n_collocation.size)

        mean_line = ax.plot(
            positions,
            steps_mean,
            "o-",
            color="#2878c8",
            linewidth=1.8,
            markersize=5.0,
            label="Média de passos",
        )[0]
        ax.set_xticks(positions)
        ax.set_xticklabels([str(value) for value in n_collocation])
        ax.set_xlabel("N_COLLOCATION")
        ax.set_ylabel("Passos até a parada (média)", color="#2878c8")
        ax.tick_params(axis="y", labelcolor="#2878c8")
        ax.grid(True, which="major", linestyle="--", alpha=0.45)
        ax.set_title(f"T = {Tfin:g}")

        variance_axis = ax.twinx()
        variance_line = variance_axis.plot(
            positions,
            steps_variance,
            "s--",
            color="#ef4a43",
            linewidth=1.5,
            markersize=4.5,
            label="Variância de passos",
        )[0]
        variance_axis.set_ylabel("Variância dos passos", color="#ef4a43")
        variance_axis.tick_params(axis="y", labelcolor="#ef4a43")

        ax.legend(
            [mean_line, variance_line],
            [mean_line.get_label(), variance_line.get_label()],
            loc="best",
            fontsize=9,
        )

    fig.suptitle(
        f"Exact CNOT-cosine HPINN: {network}, {n_layers} layer(s)",
        fontsize=14,
    )
    fig.tight_layout()
    save_figure(
        fig,
        STEP_PLOT_DIRECTORY,
        f"Steps_vs_N_COLLOCATION_{network}_layers_{n_layers}",
    )
    return True


def main() -> None:
    rows = load_saved_rows()
    summary = aggregate_rows(rows)
    save_summary(summary)

    configurations = sorted(
        {
            (str(row["network"]), int(row["n_cnot_layers"]))
            for row in summary
        }
    )

    mse_count = 0
    step_count = 0
    for network, n_layers in configurations:
        mse_count += int(plot_mse_comparison(summary, network, n_layers))
        step_count += int(plot_step_comparison(summary, network, n_layers))

    print(f"Loaded {len(rows)} saved run(s).")
    print(f"Aggregated {len(summary)} configuration point(s).")
    print(f"Generated {mse_count} MSE figure(s) and {step_count} step figure(s).")
    print(f"Comparison output: {OUTPUT_DIRECTORY}")
    print(f"Summary table: {SUMMARY_TABLE_PATH}")
    print(
        "Note: MSE yerr is the population variance by deliberate design, "
        "solely to reproduce the colleague's convention."
    )


if __name__ == "__main__":
    main()


Loaded 1800 saved run(s).
Aggregated 360 configuration point(s).
Generated 15 MSE figure(s) and 15 step figure(s).
Comparison output: /home/erickcastro/projetos/meu_projeto/CBPF_quantum_exact_colleague_comparison_uniform
Summary table: /home/erickcastro/projetos/meu_projeto/CBPF_quantum_exact_colleague_comparison_uniform/summaries/colleague_style_summary.csv
Note: MSE yerr is the population variance by deliberate design, solely to reproduce the colleague's convention.
